# CTRL01 — Representation Length Control

## Motivation

C03 narrowed the C02 representation effect sharply:

- sentence-vs-arrow encoding did **not** resolve for \(U_{\rm long}\);
- compact-vs-narrative wrapper **did** resolve under both relation encodings;
- however the compact wrapper was exactly 9 generator tokens shorter than the
  narrative wrapper.

So C03 cannot yet distinguish

\[
\boxed{\text{state wrapper / framing}}
\]

from

\[
\boxed{\text{generator input-length difference}}.
\]

C04 tests that confound directly.

---

## Fixed relation encoding

We hold the binding relation fixed as:

`LABEL is assigned to CITY.`

and compare:

1. `narr_sentence`
2. `compact_sentence`
3. `compact_sentence_lenmatch`
4. `narr_sentence_prefixctrl`
5. `cap32`

`compact_sentence_lenmatch` uses the exact same compact history as
`compact_sentence`, but enough **pre-chat neutral token IDs** are prepended to
make its complete generator input exactly the same token length as
`narr_sentence`.

The prepended tokens occur *before the whole chat*, so they shift both the
binding and generation position together and therefore do **not** change
binding-to-output distance.

`narr_sentence_prefixctrl` receives the same number of prepended neutral tokens.
It checks whether the neutral token prefix itself changes behavior.

---

# Primary contrasts

### Original wrapper effect

\[
T_{\rm compact}-T_{\rm narr}
\]

### Length-matched wrapper effect

\[
\boxed{
T_{\rm compact,lenmatch}-T_{\rm narr}
}
\]

If this remains resolved, total generator-input length cannot explain the wrapper
effect.

### Padding effect within compact representation

\[
T_{\rm compact,lenmatch}-T_{\rm compact}
\]

### Prefix-control effect within narrative representation

\[
T_{\rm narr,prefixctrl}-T_{\rm narr}
\]

If both padding effects are near zero while the length-matched wrapper contrast
remains large, the wrapper interpretation is substantially strengthened.

All directions are exploratory because C04 follows the observed C03 result.


In [ ]:
from pathlib import Path
import gc, json, random

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

cwd=Path.cwd().resolve()
ROOT=cwd.parent if cwd.name=="experiments" else cwd
OUT = ROOT / "results" / "controls" / "CTRL01_representation_length_control"
OUT.mkdir(parents=True,exist_ok=True)

GENERATOR_ID="Qwen/Qwen2.5-1.5B-Instruct"
SCORER_ID="Qwen/Qwen3-0.6B-Base"

SEED=20260901
N=60
CAP=32
WINDOWS=[8,32,128,256]
TARGET_OFFSETS=[12,20,28,36,44,52,60,68]
MAX_NEW_TOKENS=320

CONDITIONS=[
    "cap32",
    "narr_sentence",
    "compact_sentence",
    "compact_sentence_lenmatch",
    "narr_sentence_prefixctrl",
]

rng=np.random.default_rng(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

CITIES=np.array([
    "Austin","Boston","Chicago","Denver","Houston","Miami","Phoenix","Portland",
    "Raleigh","Seattle","Tucson","Atlanta","Detroit","Nashville","Orlando","Madison",
    "Buffalo","Fresno","Cleveland","Richmond","Boise","Spokane","Memphis","Baltimore",
])
ALPH=np.array(list("ABCDEFGHJKLMNPQRSTUVWXYZ"))


/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Controlled prior documents

In [2]:
PREAMBLE = """
This field manual describes routine logistics procedures used by distributed inspection
teams. Reports are written in chronological order and should remain internally consistent.
Each team reviews equipment condition, transport constraints, weather, communications,
access permissions, and the order of routine checks. Inspectors record observations
without embellishment and distinguish confirmed information from assumptions. When an
assignment is given earlier in a report, later sections should preserve that assignment
unless an explicit reassignment is recorded. Reports use concise technical prose and avoid
unnecessary restatement. Site visits involve arrival checks, an equipment survey,
verification of local constraints, documentation of anomalies, and a closing note
describing what should happen next. Communications are logged separately from observations
so later readers can reconstruct the sequence of decisions. Equipment identifiers,
assigned destinations, and status changes are treated as factual state that carries
forward through the report. Before departure, teams verify cases, radios, batteries,
weather covers, transport documents, and local access requirements. Routine inspection
details should not overwrite earlier operational assignments.
""".strip()

BRIDGE = """
After the assignments were recorded, the team completed an unrelated equipment inspection.
Battery housings were checked, cable connectors were reseated, protective cases were
counted, and backup power modules were verified. These routine checks did not modify any
earlier assignment.
""".strip()

def rlabel():
    return "".join(rng.choice(ALPH,size=5,replace=True).tolist())

def rcode(n):
    chars=np.array(list("ABCDEFGHJKLMNPQRSTUVWXYZ23456789"))
    return "".join(rng.choice(chars,size=n,replace=True).tolist())

def make_tail():
    return (
        f"Administrative reference {rcode(7)} was logged for this visit. "
        f"Inventory listed {int(rng.integers(6,30))} seals, "
        f"{int(rng.integers(5,25))} weather covers, "
        f"{int(rng.integers(8,40))} numbered tags, and "
        f"{int(rng.integers(4,18))} protective cases. "
        f"The radio test code was {rcode(4)}. "
        "These bookkeeping details are unrelated to destination assignments and "
        "do not modify earlier operational state."
    )

def sentence_lines(bindings):
    return "\n".join(
        f"{lab} is assigned to {city}."
        for lab,city in bindings.items()
    )

def narr_block(bindings):
    return (
        "The following four assignments are now fixed:\n"
        + sentence_lines(bindings)
        + "\nThese assignments remain in force for the rest of the report."
    )

def compact_block(bindings):
    return (
        "[COMPACT STATE]\n"
        + sentence_lines(bindings)
        + "\n[END COMPACT STATE]"
    )

def canonical_raw_block(bindings):
    return (
        "The following four assignments are now fixed:\n"
        + "\n".join(f"{lab} -> {city}" for lab,city in bindings.items())
        + "\nThese assignments remain in force for the rest of the report."
    )

design=[]
for i in range(N):
    labels=[]
    while len(labels)<4:
        x=rlabel()
        if x not in labels:
            labels.append(x)

    cities=rng.choice(CITIES,size=4,replace=False).tolist()
    b=dict(zip(labels,cities))
    tail=make_tail()

    narr=narr_block(b)
    compact=compact_block(b)
    canonical=canonical_raw_block(b)

    # Scorer always sees one fixed canonical document.
    original_prior=PREAMBLE+"\n\n"+canonical+"\n\n"+BRIDGE+"\n\n"+tail

    design.append({
        "instance_id":i,
        "bindings":b,
        "tail":tail,
        "narr":narr,
        "compact":compact,
        "original_prior":original_prior,
    })

display(pd.DataFrame([{
    "instance_id":d["instance_id"],
    "bindings":json.dumps(d["bindings"]),
    "tail":d["tail"][:80]+"..."
} for d in design[:5]]))


,instance_id,bindings,tail
0,0,"{""URVPU"": ""Baltimore"", ""JYGAD"": ""Memphis"", ""JA...",Administrative reference RHS9SGX was logged fo...
1,1,"{""ELGBL"": ""Orlando"", ""GHTWF"": ""Denver"", ""RDPKF...",Administrative reference Z3GJ5P9 was logged fo...
2,2,"{""RRDJF"": ""Houston"", ""LHTFC"": ""Detroit"", ""TEMD...",Administrative reference WLC7HJ4 was logged fo...
3,3,"{""ZCBAU"": ""Seattle"", ""HJVNA"": ""Raleigh"", ""WPQQ...",Administrative reference 9UKKFAW was logged fo...
4,4,"{""DQTUJ"": ""Orlando"", ""KEYBS"": ""Seattle"", ""XCKZ...",Administrative reference 8A9PL9H was logged fo...


## 2. Generator histories and exact token-length matching

In [3]:
gtok=AutoTokenizer.from_pretrained(GENERATOR_ID,trust_remote_code=True)
gmodel=AutoModelForCausalLM.from_pretrained(
    GENERATOR_ID,torch_dtype="auto",device_map="auto",trust_remote_code=True
)
gmodel.eval()

SYSTEM_USER=(
    "You are maintaining a field report. Treat the previous assistant message as the "
    "report written so far and answer or continue from it consistently."
)

def enc(text):
    return gtok(text,add_special_tokens=False,return_tensors="pt").input_ids[0].cpu()

def last32(text):
    ids=enc(text)
    return gtok.decode(ids[-CAP:],skip_special_tokens=True)

def histories(d):
    narr=PREAMBLE+"\n\n"+d["narr"]+"\n\n"+BRIDGE+"\n\n"+d["tail"]
    compact=PREAMBLE+"\n\n"+d["compact"]+"\n\n"+BRIDGE+"\n\n"+d["tail"]
    return {
        "narr_sentence":narr,
        "compact_sentence":compact,
        "cap32":last32(narr),
    }

# Fixed benign prefix-token reservoir. We prepend exact token IDs from this reservoir
# before the chat sequence, so matching is exact and binding-to-output distance is unchanged.
PAD_RESERVOIR_TEXT=(
    "Administrative note. Routine context unchanged. Reference information only. "
    "No operational assignment is modified. "
)*8
PAD_RESERVOIR_IDS=enc(PAD_RESERVOIR_TEXT)

def base_chat_ids(history,query):
    msgs=[
        {"role":"user","content":SYSTEM_USER},
        {"role":"assistant","content":history},
        {"role":"user","content":query},
    ]
    x=gtok.apply_chat_template(
        msgs,tokenize=True,add_generation_prompt=True,return_tensors="pt"
    )
    x=x.input_ids if hasattr(x,"input_ids") else x
    return x[0].cpu()

def prepend_pad(ids,n):
    if n<=0:
        return ids.clone()
    assert n<=len(PAD_RESERVOIR_IDS)
    return torch.cat([PAD_RESERVOIR_IDS[:n],ids],dim=0)

geom=[]
caps=[]

DUMMY_QUERY="Continue the report."

for d in design:
    h=histories(d)
    narr_ids=base_chat_ids(h["narr_sentence"],DUMMY_QUERY)
    compact_ids=base_chat_ids(h["compact_sentence"],DUMMY_QUERY)
    delta=len(narr_ids)-len(compact_ids)

    caps.append(h["cap32"])

    geom.append({
        "instance_id":d["instance_id"],
        "narr_input_tokens":len(narr_ids),
        "compact_input_tokens":len(compact_ids),
        "tokens_needed_to_match":delta,
        "memory_end_to_cont":len(enc(BRIDGE+"\n\n"+d["tail"])),
        "tail_tokens":len(enc(d["tail"])),
    })

geom=pd.DataFrame(geom)
display(geom.describe().T)

print("unique CAP32 strings:",len(set(caps)),"/",N)
print("unique match deltas:",sorted(geom["tokens_needed_to_match"].unique().tolist()))

assert len(set(caps))==N
assert (geom["tokens_needed_to_match"]>0).all()
assert (geom["memory_end_to_cont"]>32).all()


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 338/338 [00:00<00:00, 698.42it/s]


,count,mean,std,min,25%,50%,75%,max
instance_id,60.0,29.500000,17.464249,0.0,14.75,29.5,44.25,59.0
narr_input_tokens,60.0,447.583333,1.871206,442.0,446.00,447.5,449.00,451.0
compact_input_tokens,60.0,438.583333,1.871206,433.0,437.00,438.5,440.00,442.0
tokens_needed_to_match,60.0,9.000000,0.000000,9.0,9.00,9.0,9.00,9.0
memory_end_to_cont,60.0,119.050000,1.477802,115.0,118.00,119.0,120.00,122.0
tail_tokens,60.0,66.050000,1.477802,62.0,65.00,66.0,67.00,69.0


unique CAP32 strings: 60 / 60
unique match deltas: [9]


## 3. Build condition-specific exact inputs

For each query and instance:

- `narr_sentence`: unmodified narrative chat input.
- `compact_sentence`: unmodified compact chat input.
- `compact_sentence_lenmatch`: prepend exactly enough neutral token IDs to equal
  the narrative input length for **that exact query**.
- `narr_sentence_prefixctrl`: prepend the same number of neutral IDs to narrative.
- `cap32`: truncated local history.

The notebook asserts exact length equality:

\[
|\text{compact\_sentence\_lenmatch}|
=
|\text{narr\_sentence}|.
\]


In [4]:
def condition_input_ids(d,query,condition):
    h=histories(d)

    narr=base_chat_ids(h["narr_sentence"],query)
    compact=base_chat_ids(h["compact_sentence"],query)
    cap=base_chat_ids(h["cap32"],query)

    delta=len(narr)-len(compact)
    assert delta>0

    if condition=="narr_sentence":
        return narr,delta

    if condition=="compact_sentence":
        return compact,delta

    if condition=="compact_sentence_lenmatch":
        out=prepend_pad(compact,delta)
        assert len(out)==len(narr)
        return out,delta

    if condition=="narr_sentence_prefixctrl":
        return prepend_pad(narr,delta),delta

    if condition=="cap32":
        return cap,delta

    raise ValueError(condition)

def greedy(ids,max_new_tokens):
    ids=ids.reshape(1,-1).to(gmodel.device)
    with torch.inference_mode():
        y=gmodel.generate(
            input_ids=ids,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=gtok.eos_token_id,
            eos_token_id=gtok.eos_token_id,
            use_cache=True,
        )
    new=y[0,ids.shape[1]:]
    return gtok.decode(new,skip_special_tokens=True).strip(),new.detach().cpu()

# Exact audit on the continuation-query form will be repeated after QUERY is defined.


## 4. Manipulation check

In [5]:
PROBE_PATH=OUT/"binding_probes.parquet"

existing=pd.read_parquet(PROBE_PATH) if PROBE_PATH.exists() else pd.DataFrame()
done=set(zip(existing["instance_id"],existing["condition"],existing["label"])) if len(existing) else set()
rows=existing.to_dict("records") if len(existing) else []

for d in tqdm(design,desc="C04 probes"):
    for condition in CONDITIONS:
        for lab,city in d["bindings"].items():
            key=(d["instance_id"],condition,lab)
            if key in done:
                continue

            q=f"What destination was assigned to {lab}? Reply with the destination city only."
            ids,delta=condition_input_ids(d,q,condition)
            ans,new=greedy(ids,12)

            rows.append({
                "instance_id":d["instance_id"],
                "condition":condition,
                "label":lab,
                "expected_city":city,
                "answer":ans,
                "correct":city.lower() in ans.lower(),
                "prefix_pad_tokens":(
                    delta if condition in ["compact_sentence_lenmatch","narr_sentence_prefixctrl"] else 0
                ),
                "input_tokens":len(ids),
                "answer_tokens":len(new),
            })

    pd.DataFrame(rows).to_parquet(PROBE_PATH,index=False)

probes=pd.read_parquet(PROBE_PATH)

Z=(
    probes.groupby(["instance_id","condition"])["correct"]
    .mean().rename("Z_binding").reset_index()
)

display(Z.groupby("condition")["Z_binding"].agg(["mean","std","count"]))
display(probes[probes["instance_id"]==0][[
    "condition","label","expected_city","answer","correct","input_tokens","prefix_pad_tokens"
]])


C04 probes: 100%|███████████████████████████████████████████████████████████████████████████████████████| 60/60 [03:05<00:00,  3.10s/it]


,mean,std,count
condition,,,
cap32,0.000000,0.000000,60
compact_sentence,0.950000,0.100844,60
compact_sentence_lenmatch,0.950000,0.100844,60
narr_sentence,0.995833,0.032275,60
narr_sentence_prefixctrl,0.991667,0.045255,60


,condition,label,expected_city,answer,correct,input_tokens,prefix_pad_tokens
0,cap32,URVPU,Baltimore,URVPU was assigned to DFW (Dallas/Fort,False,113,0
1,cap32,JYGAD,Memphis,JYGAD was assigned to Guangzhou.,False,113,0
2,cap32,JAPEV,Buffalo,The destination for JAPEV is unspecified in th...,False,113,0
3,cap32,LPBTZ,Nashville,"I'm sorry, but I don't have enough information to",False,113,0
4,narr_sentence,URVPU,Baltimore,Baltimore,True,457,0
5,narr_sentence,JYGAD,Memphis,Memphis,True,457,0
6,narr_sentence,JAPEV,Buffalo,Buffalo,True,457,0
7,narr_sentence,LPBTZ,Nashville,Nashville,True,457,0
8,compact_sentence,URVPU,Baltimore,Baltimore,True,448,0
9,compact_sentence,JYGAD,Memphis,Memphis,True,448,0


## STOP GATE

Proceed only if all four memory-bearing conditions have comparably high direct
binding recall and CAP32 remains near zero.

In particular, neutral prefix padding itself must not materially degrade
\(Z_{\rm binding}\).


## 5. Natural continuation branch

In [6]:
GEN_PATH=OUT/"natural_continuations.parquet"

QUERY=(
    "Continue the field report in exactly six sentences. Each sentence must contain "
    "no more than 25 words. Return to the previously assigned items and discuss their "
    "next operational steps consistently with the assignments already recorded. "
    "Integrate relevant locations naturally where useful. Do not reproduce the assignment "
    "table, do not use arrow notation, and do not describe this as a memory test. "
    "Stop after the sixth sentence."
)

# Exact continuation-query length audit.
audit=[]
for d in design:
    narr,_=condition_input_ids(d,QUERY,"narr_sentence")
    compact,_=condition_input_ids(d,QUERY,"compact_sentence")
    matched,delta=condition_input_ids(d,QUERY,"compact_sentence_lenmatch")
    narr_ctrl,_=condition_input_ids(d,QUERY,"narr_sentence_prefixctrl")

    assert len(matched)==len(narr)

    audit.append({
        "instance_id":d["instance_id"],
        "narr_tokens":len(narr),
        "compact_tokens":len(compact),
        "compact_matched_tokens":len(matched),
        "narr_prefixctrl_tokens":len(narr_ctrl),
        "delta":delta,
    })

audit=pd.DataFrame(audit)
display(audit.describe().T)
assert (audit["compact_matched_tokens"]==audit["narr_tokens"]).all()

existing=pd.read_parquet(GEN_PATH) if GEN_PATH.exists() else pd.DataFrame()
done=set(zip(existing["instance_id"],existing["condition"])) if len(existing) else set()
rows=existing.to_dict("records") if len(existing) else []

for d in tqdm(design,desc="C04 continuations"):
    for condition in CONDITIONS:
        key=(d["instance_id"],condition)
        if key in done:
            continue

        ids,delta=condition_input_ids(d,QUERY,condition)
        cont,new=greedy(ids,MAX_NEW_TOKENS)

        rows.append({
            "instance_id":d["instance_id"],
            "condition":condition,
            "bindings_json":json.dumps(d["bindings"]),
            "original_prior":d["original_prior"],
            "continuation_text":cont,
            "scored_document":d["original_prior"]+"\n\n"+cont,
            "input_tokens":len(ids),
            "prefix_pad_tokens":(
                delta if condition in ["compact_sentence_lenmatch","narr_sentence_prefixctrl"] else 0
            ),
            "continuation_tokens":len(new),
            "hit_generation_cap":bool(len(new)>=MAX_NEW_TOKENS),
        })

    pd.DataFrame(rows).to_parquet(GEN_PATH,index=False)

gens=pd.read_parquet(GEN_PATH)

display(gens.groupby("condition")[["input_tokens","continuation_tokens"]].agg(["mean","std","count"]))
display(gens.groupby("condition")["hit_generation_cap"].mean().rename("hit_cap_fraction").to_frame())
display(gens.groupby("condition")["continuation_text"].nunique().rename("n_unique").to_frame())


,count,mean,std,min,25%,50%,75%,max
instance_id,60.0,29.500000,17.464249,0.0,14.75,29.5,44.25,59.0
narr_tokens,60.0,520.583333,1.871206,515.0,519.00,520.5,522.00,524.0
compact_tokens,60.0,511.583333,1.871206,506.0,510.00,511.5,513.00,515.0
compact_matched_tokens,60.0,520.583333,1.871206,515.0,519.00,520.5,522.00,524.0
narr_prefixctrl_tokens,60.0,529.583333,1.871206,524.0,528.00,529.5,531.00,533.0
delta,60.0,9.000000,0.000000,9.0,9.00,9.0,9.00,9.0


C04 continuations: 100%|████████████████████████████████████████████████████████████████████████████████| 60/60 [21:58<00:00, 21.97s/it]


input_tokens                 continuation_tokens  \
                                  mean       std count                mean   
condition                                                                    
cap32                       174.000000  0.000000    60          175.083333   
compact_sentence            511.583333  1.871206    60          164.483333   
compact_sentence_lenmatch   520.583333  1.871206    60          191.133333   
narr_sentence               520.583333  1.871206    60          236.916667   
narr_sentence_prefixctrl    529.583333  1.871206    60          210.783333   

                                            
                                 std count  
condition                                   
cap32                      53.807029    60  
compact_sentence           64.517634    60  
compact_sentence_lenmatch  72.640621    60  
narr_sentence              77.896126    60  
narr_sentence_prefixctrl   83.108892    60

,hit_cap_fraction
condition,
cap32,0.000000
compact_sentence,0.016667
compact_sentence_lenmatch,0.100000
narr_sentence,0.266667
narr_sentence_prefixctrl,0.200000


,n_unique
condition,
cap32,59
compact_sentence,60
compact_sentence_lenmatch,60
narr_sentence,60
narr_sentence_prefixctrl,60


## 6. Natural binding-use diagnostic

In [7]:
def natural_use(row):
    b=json.loads(row["bindings_json"])
    text=row["continuation_text"].lower()
    return float(np.mean([city.lower() in text for city in b.values()]))

gens["natural_binding_use"]=gens.apply(natural_use,axis=1)
gens.to_parquet(GEN_PATH,index=False)

display(gens.groupby("condition")["natural_binding_use"].agg(["mean","std","count"]))


,mean,std,count
condition,,,
cap32,0.000000,0.000000,60
compact_sentence,0.766667,0.381974,60
compact_sentence_lenmatch,0.666667,0.435857,60
narr_sentence,0.912500,0.247123,60
narr_sentence_prefixctrl,0.858333,0.313424,60


## 7. Frozen scorer

In [8]:
del gmodel
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

stok=AutoTokenizer.from_pretrained(SCORER_ID,trust_remote_code=True)
smodel=AutoModelForCausalLM.from_pretrained(
    SCORER_ID,torch_dtype="auto",device_map="auto",trust_remote_code=True
)
smodel.eval()

if stok.pad_token_id is None:
    stok.pad_token_id=stok.eos_token_id

SCORE_PATH=OUT/"scores.parquet"

def score_doc(document,prior):
    ids=stok(document,add_special_tokens=False).input_ids
    pids=stok(prior,add_special_tokens=False).input_ids
    start=len(pids)

    targets=[start+o for o in TARGET_OFFSETS if start+o<len(ids)]
    if len(targets)!=len(TARGET_OFFSETS):
        return None

    ctxs,tgts,ks=[],[],[]

    for t in targets:
        for k in WINDOWS:
            if t<k:
                return None
            ctxs.append(torch.tensor(ids[t-k:t],dtype=torch.long))
            tgts.append(ids[t])
            ks.append(k)

    L=max(map(len,ctxs))
    batch=torch.full((len(ctxs),L),stok.pad_token_id,dtype=torch.long)
    mask=torch.zeros_like(batch)

    for i,x in enumerate(ctxs):
        batch[i,-len(x):]=x
        mask[i,-len(x):]=1

    batch=batch.to(smodel.device)
    mask=mask.to(smodel.device)
    tgt=torch.tensor(tgts,dtype=torch.long,device=smodel.device)

    with torch.inference_mode():
        logits=smodel(input_ids=batch,attention_mask=mask,use_cache=False).logits[:,-1,:]
        lp=torch.log_softmax(logits.float(),dim=-1)
        losses=-lp[torch.arange(len(tgt),device=smodel.device),tgt]

    losses=losses.detach().cpu().numpy()

    out={}
    for k in WINDOWS:
        out[f"L_{k}"]=float(np.mean([v for v,kk in zip(losses,ks) if kk==k]))

    out["U_32"]=out["L_8"]-out["L_32"]
    out["U_128"]=out["L_8"]-out["L_128"]
    out["U_256"]=out["L_8"]-out["L_256"]
    out["U_long"]=0.5*(out["U_128"]+out["U_256"])
    out["selectivity"]=out["U_long"]-out["U_32"]
    return out

existing=pd.read_parquet(SCORE_PATH) if SCORE_PATH.exists() else pd.DataFrame()
done=set(zip(existing["instance_id"],existing["condition"])) if len(existing) else set()
rows=existing.to_dict("records") if len(existing) else []

for _,r in tqdm(gens.iterrows(),total=len(gens),desc="C04 scoring"):
    key=(int(r["instance_id"]),str(r["condition"]))
    if key in done:
        continue

    s=score_doc(r["scored_document"],r["original_prior"])

    rows.append({
        "instance_id":int(r["instance_id"]),
        "condition":r["condition"],
        "eligible":s is not None,
        **({} if s is None else s),
    })

    pd.DataFrame(rows).to_parquet(SCORE_PATH,index=False)

scores=pd.read_parquet(SCORE_PATH)
display(scores.groupby("condition")["eligible"].agg(["sum","count"]))


C04 scoring: 100%|████████████████████████████████████████████████████████████████████████████████████| 300/300 [04:03<00:00,  1.23it/s]


,sum,count
condition,,
cap32,59,60
compact_sentence,60,60
compact_sentence_lenmatch,60,60
narr_sentence,60,60
narr_sentence_prefixctrl,60,60


## 8. Joint paired analysis

In [9]:
analysis=(
    gens.merge(Z,on=["instance_id","condition"],how="left")
    .merge(scores,on=["instance_id","condition"],how="left")
)

elig=analysis.pivot_table(
    index="instance_id",
    columns="condition",
    values="eligible",
    aggfunc="first",
).fillna(False)

joint=elig.index[elig[CONDITIONS].all(axis=1)]
a=analysis[analysis["instance_id"].isin(joint)].copy()

print("jointly eligible:",len(joint),"/",N)

display(a.groupby("condition")[[
    "Z_binding",
    "natural_binding_use",
    "input_tokens",
    "continuation_tokens",
    "U_32","U_128","U_256","U_long","selectivity"
]].agg(["mean","std","count"]))


jointly eligible: 59 / 60


Z_binding                 natural_binding_use  \
                               mean       std count                mean   
condition                                                                 
cap32                      0.000000  0.000000    59            0.000000   
compact_sentence           0.949153  0.101494    59            0.762712   
compact_sentence_lenmatch  0.949153  0.101494    59            0.661017   
narr_sentence              0.995763  0.032547    59            0.911017   
narr_sentence_prefixctrl   0.991525  0.045630    59            0.868644   

                                          input_tokens                  \
                                std count         mean       std count   
condition                                                                
cap32                      0.000000    59   174.000000  0.000000    59   
compact_sentence           0.384012    59   511.610169  1.875587    59   
compact_sentence_lenmatch  0.437377    59   520.610169  1.875587    59   
narr_sentence              0.248975    59   520.610169  1.875587    59   
narr_sentence_prefixctrl   0.305679    59   529.610169  1.875587    59   

                          continuation_tokens  ... U_128     U_256            \
                                         mean  ... count      mean       std   
condition                                      ...                             
cap32                              176.949153  ...    59  1.109471  0.748816   
compact_sentence                   164.881356  ...    59  2.998961  1.075445   
compact_sentence_lenmatch          191.932203  ...    59  3.172766  1.259973   
narr_sentence                      237.796610  ...    59  2.913876  0.923675   
narr_sentence_prefixctrl           210.661017  ...    59  3.032655  0.953801   

                                   U_long                 selectivity  \
                          count      mean       std count        mean   
condition                                                               
cap32                        59  1.132453  0.755741    59    0.326433   
compact_sentence             59  2.305480  0.768704    59    1.241021   
compact_sentence_lenmatch    59  2.491924  0.919475    59    1.406891   
narr_sentence                59  2.592015  0.748646    59    1.816846   
narr_sentence_prefixctrl     59  2.637160  0.756819    59    1.844241   

                                           
                                std count  
condition                                  
cap32                      0.498572    59  
compact_sentence           0.687452    59  
compact_sentence_lenmatch  0.834328    59  
narr_sentence              0.721234    59  
narr_sentence_prefixctrl   0.728434    59  

[5 rows x 27 columns]

## 9. Planned paired contrasts

In [10]:
def wide(metric):
    return a.pivot(index="instance_id",columns="condition",values=metric).dropna()

def boot(metric,left,right,B=10000):
    w=wide(metric)
    d=(w[left]-w[right]).to_numpy(float)
    rr=np.random.default_rng(SEED)
    bs=np.empty(B)

    for i in range(B):
        ix=rr.integers(0,len(d),len(d))
        bs[i]=d[ix].mean()

    return {
        "metric":metric,
        "contrast":f"{left} - {right}",
        "n":len(d),
        "estimate":float(d.mean()),
        "ci_low":float(np.quantile(bs,.025)),
        "ci_high":float(np.quantile(bs,.975)),
    }

PLANNED=[
    ("compact_sentence","narr_sentence"),
    ("compact_sentence_lenmatch","narr_sentence"),
    ("compact_sentence_lenmatch","compact_sentence"),
    ("narr_sentence_prefixctrl","narr_sentence"),
    ("compact_sentence_lenmatch","narr_sentence_prefixctrl"),
    ("cap32","narr_sentence"),
]

METRICS=[
    "Z_binding",
    "natural_binding_use",
    "input_tokens",
    "continuation_tokens",
    "U_32","U_128","U_256","U_long","selectivity",
]

result=pd.DataFrame([
    boot(m,l,r)
    for m in METRICS
    for l,r in PLANNED
])

display(result[result["metric"].isin([
    "Z_binding","input_tokens","U_long","selectivity"
])])

result.to_csv(OUT/"paired_contrasts.csv",index=False)


,metric,contrast,n,estimate,ci_low,ci_high
0,Z_binding,compact_sentence - narr_sentence,59,-0.046610,-0.072034,-0.025424
1,Z_binding,compact_sentence_lenmatch - narr_sentence,59,-0.046610,-0.072034,-0.025424
2,Z_binding,compact_sentence_lenmatch - compact_sentence,59,0.000000,0.000000,0.000000
3,Z_binding,narr_sentence_prefixctrl - narr_sentence,59,-0.004237,-0.012712,0.000000
4,Z_binding,compact_sentence_lenmatch - narr_sentence_pref...,59,-0.042373,-0.067797,-0.021186
5,Z_binding,cap32 - narr_sentence,59,-0.995763,-1.000000,-0.987288
12,input_tokens,compact_sentence - narr_sentence,59,-9.000000,-9.000000,-9.000000
13,input_tokens,compact_sentence_lenmatch - narr_sentence,59,0.000000,0.000000,0.000000
14,input_tokens,compact_sentence_lenmatch - compact_sentence,59,9.000000,9.000000,9.000000
15,input_tokens,narr_sentence_prefixctrl - narr_sentence,59,9.000000,9.000000,9.000000


# Interpretation

## Wrapper survives exact length matching

If

\[
U_{\rm compact,lenmatch}
-
U_{\rm narr}
\]

remains resolved while:

- \(Z_{\rm binding}\) is matched,
- total generator input length is exactly matched,
- compact padding itself has little effect,
- narrative prefix-control padding has little effect,

then the C03 wrapper effect cannot be reduced to a 9-token input-length
difference.

That would support a generation-side **state framing / packaging** mechanism.

## Length explains the effect

If the original compact-vs-narrative difference appears, but disappears after
exact input-length matching, then the C03 wrapper result was largely a context
length artifact.

## Prefix tokens themselves matter

If `narr_sentence_prefixctrl - narr_sentence` is large, the neutral prefix is
not inert enough and the length-control construction is itself contaminated.
Do not interpret the length-matched wrapper contrast.

## Scope

C04 does not establish why compact framing changes later dependency organization.
It only tests whether the C03 wrapper effect survives an exact total-input-length
control. A positive result would motivate later mechanisms such as state
salience, explicit memory packaging, or planning behavior.
